# Scientific Graphing Lab – CSV-Diagramme

1. Lade deine CSV-Datei in **denselben Ordner** wie dieses Notebook hoch (Upload-Pfeil im Dateibrowser).
2. Trage unten bei `filename` den **exakten Namen** deiner CSV-Datei ein.
3. Ändere Diagrammtyp, Farbe, Titel oder Achsenbereiche **direkt im Code**.
4. Führe die Zelle mit **Shift + Enter** aus.
5. Tippe unter dem Diagramm auf **„Diagramm herunterladen (PNG)“**, um die Grafik direkt auf deinem Gerät zu speichern.
6. Lade dein bearbeitetes Notebook am Stundenende herunter, wenn du den Code behalten möchtest.

**CSV-Format:** Erste Zeile = Spaltenüberschriften; danach zwei numerische Spalten (x und y). Komma, Semikolon oder Tab als Trennzeichen sind möglich. Bei Semikolondateien werden deutsche Dezimalkommas unterstützt.

**Hinweis:** Die Spaltenüberschriften werden automatisch als Achsenbeschriftungen verwendet. Du kannst sie unten bei `x_label` und `y_label` überschreiben. Für die automatische Achsenskalierung `None` stehen lassen.

**Originalcode beschädigt?** Lade die unveränderte Masterdatei erneut aus dem GitHub-Repository herunter. Ein bloßes Neuladen der Website stellt die Vorlage nicht unbedingt wieder her.


In [ ]:
import csv
import base64
import io
from html import escape
from IPython.display import HTML, display
import matplotlib.pyplot as plt

# ===== HIER DÜRFEN SCHÜLERINNEN UND SCHÜLER EXPERIMENTIEREN =====
filename = "katalase_messwerte.csv"  # Dateiname deiner hochgeladenen CSV

graph_type = "scatter"  # "scatter", "line" oder "bar"
colour = "blue"         # z. B. "red", "green", "orange"
graph_title = "Mein Experiment"
show_grid = True
point_size = 60
output_filename = "mein_diagramm.png"  # Dateiname beim Herunterladen

# None = automatisch passend zu den Daten; Zahl = manuelle Grenze
x_min, x_max = None, None
y_min, y_max = None, None

# None = Spaltenüberschrift der CSV verwenden
x_label = None
y_label = None

# ===== CSV EINLESEN: ERSTE SPALTE = X, ZWEITE SPALTE = Y =====
with open(filename, encoding="utf-8-sig", newline="") as file:
    sample = file.read(4096)
    file.seek(0)
    dialect = csv.Sniffer().sniff(sample, delimiters=",;\t")
    reader = csv.reader(file, dialect=dialect)
    header = next(reader, None)
    if header is None or len(header) != 2:
        raise ValueError("Die CSV muss genau zwei Spalten mit Überschriften haben.")
    rows = [row for row in reader if row and any(cell.strip() for cell in row)]

if not rows:
    raise ValueError("Die CSV enthält keine Messwerte.")

x, y = [], []
for line_number, row in enumerate(rows, start=2):
    if len(row) != 2:
        raise ValueError(f"Zeile {line_number}: Erwartet werden zwei Werte.")
    try:
        x.append(float(row[0].strip().replace(",", ".")))
        y.append(float(row[1].strip().replace(",", ".")))
    except ValueError as error:
        raise ValueError(f"Zeile {line_number}: Beide Werte müssen Zahlen sein.") from error

# ===== DIAGRAMM ERSTELLEN =====
fig, ax = plt.subplots(figsize=(8, 5))
if graph_type == "scatter":
    ax.scatter(x, y, color=colour, s=point_size)
elif graph_type == "line":
    ax.plot(x, y, color=colour, marker="o")
elif graph_type == "bar":
    ax.bar(x, y, color=colour)
else:
    raise ValueError("graph_type muss 'scatter', 'line' oder 'bar' sein.")

ax.set_xlabel(x_label if x_label is not None else header[0].strip())
ax.set_ylabel(y_label if y_label is not None else header[1].strip())
ax.set_title(graph_title)

# Automatische Grenzen bleiben erhalten, solange None eingestellt ist.
if x_min is not None or x_max is not None:
    ax.set_xlim(left=x_min, right=x_max)
if y_min is not None or y_max is not None:
    ax.set_ylim(bottom=y_min, top=y_max)

ax.grid(show_grid, alpha=0.3)
fig.tight_layout()
# ===== PNG ERZEUGEN UND DIREKTEN DOWNLOAD-LINK ANZEIGEN =====
# Die Grafik wird als PNG im Arbeitsspeicher erzeugt.
# Der Download-Link funktioniert per Fingertipp auch auf Tablets.
image_buffer = io.BytesIO()
fig.savefig(image_buffer, format="png", dpi=300, bbox_inches="tight")
image_buffer.seek(0)
image_base64 = base64.b64encode(image_buffer.read()).decode("ascii")

plt.show()

# Kein Rechtsklick im JupyterLite-Dateibrowser erforderlich.
# Der Browser kann nachfragen, ob die Datei gespeichert werden soll.
safe_name = escape(output_filename, quote=True)
display(HTML(
    f'<a download="{safe_name}" '
    f'href="data:image/png;base64,{image_base64}" '
    'style="display:inline-block;padding:12px 18px;'
    'background:#1769aa;color:white;text-decoration:none;'
    'border-radius:8px;font-weight:bold">'
    'Diagramm herunterladen (PNG)</a>'
))
